# Download OpenET Data via Google Earth Engine

Downloads bimonthly median ET for Iowa over 2015–2020.

**Workflow:**
1. Load OpenET daily ensemble from GEE
2. Mask water bodies, low-quality pixels, and implausible values
3. Aggregate to bimonthly medians (1–15th and 16th–end of month)
4. Reproject/resample to match a user-uploaded reference raster
5. Export each time step as a GeoTIFF to Google Drive

**Prerequisites:**
- A Google Earth Engine account: https://earthengine.google.com/
- `earthengine-api` installed: `pip install earthengine-api`
- Run `earthengine authenticate` once in a terminal to set up credentials
- Your reference raster uploaded to GEE Assets (see Section 4 below)


Sophie Ruehr
sophie.ruehr@gmail.com
Feb 25 2026

---
## Section 0 — Authenticate & Initialize

In [1]:
import ee
import datetime
import calendar
import time
from collections import Counter

# ── First time only: uncomment and run this line to authenticate ──
# ee.Authenticate()

# Replace 'your-project-id' with your GEE Cloud project ID.
# Find it at: https://console.cloud.google.com/
ee.Initialize(project='your-project-id')
print('Earth Engine initialized successfully.')

ModuleNotFoundError: No module named 'ee'

---
## Section 1 — Configuration

**Edit the values in this cell.** Everything else should run without changes.

In [ ]:
# ── Years to process ──────────────────────────────────────────────────────────
YEARS = list(range(2015, 2021))   # 2015 through 2020 inclusive, adjust as needed

# ── OpenET GEE collection IDs ─────────────────────────────────────────────────
# Daily ensemble (needed for bimonthly aggregation) - this is the mean of all models in the OpenET ensemble, not a single model:
# Check out details here: https://developers.google.com/earth-engine/datasets/catalog/OpenET_ENSEMBLE_CONUS_GRIDMET_MONTHLY_v2_0
OPENET_DAILY   = 'OpenET/ENSEMBLE/CONUS/GRIDMET/DAILY/v2_0'

# ── Reference raster (sets target CRS + pixel grid) ─── ────────────────────────
# Upload a SIF raster that covers your study area, with the desired CRS and pixel size. 
# This will be used as the target grid for reprojection and resampling of the OpenET data.
# To upload your reference .tif to GEE as an asset:
#   Assets tab → New → Image upload  (in the GEE Code Editor)
# Then paste the asset path here, e.g.:
REFERENCE_ASSET = 'projects/your-project-id/assets/your_reference_raster'

# ── Google Drive export folder ────────────────────────────────────────────────
EXPORT_FOLDER = 'OpenET_Iowa_Bimonthly'

# ── Band names in the OpenET daily ensemble ───────────────────────────────────
# Inspect the actual band names in Section 2 below before assuming these.
ET_BAND    = 'et_ensemble_mad'       # mean ET estimate (mm/day)
COUNT_BAND = 'et_ensemble_mad_count'    # number of models contributing; a proxy for quality

# ── Quality filter: minimum number of models that must agree ──────────────────
# OpenET ensemble uses up to 6 models. Requiring ≥3 keeps reasonably confident
# pixels while retaining good spatial coverage.
MIN_MODEL_COUNT = 3

---
## Section 2 — Inspect the OpenET Collection

Run this first to confirm band names, date range, and image count before processing.

In [ ]:
# Load and inspect the daily ensemble collection
openet_col = ee.ImageCollection(OPENET_DAILY)

# Grab one image for inspection
sample = openet_col.first()

print('=== OpenET Daily Ensemble ===')
print('Band names  :', sample.bandNames().getInfo())
print('Properties  :', sample.propertyNames().getInfo())
print('First image :', ee.Date(sample.get('system:time_start')).format('YYYY-MM-dd').getInfo())

# Check how many images are available over Iowa for 2015–2020
iowa = ee.FeatureCollection('TIGER/2018/States') \
         .filter(ee.Filter.eq('NAME', 'Iowa')).geometry()

n_images = openet_col.filterBounds(iowa) \
                     .filterDate('2015-01-01', '2021-01-01') \
                     .size().getInfo()
print(f'Images over Iowa (2015–2020): {n_images}')

# Print scale/resolution of the native data
print('Native scale (m):', sample.select(0).projection().nominalScale().getInfo())

---
## Section 3 — Study Area (Iowa)

In [ ]:
# Load Iowa state boundary from the US Census TIGER dataset
states = ee.FeatureCollection('TIGER/2018/States')
iowa   = states.filter(ee.Filter.eq('NAME', 'Iowa')).geometry()

# Quick sanity check
print('Iowa bounds:', iowa.bounds().getInfo()['coordinates'])

---
## Section 4 — Load Reference Image for Reprojection (here SIF raster)

**How to upload a reference raster to GEE:**
1. Go to https://code.earthengine.google.com/
2. In the left panel, click **Assets** → **New** → **Image upload**
3. Upload your `.tif` file and note the asset path
4. Paste the path into `REFERENCE_ASSET` in Section 1

The `crs` and `crsTransform` extracted here are used at export time to snap
the output exactly onto your reference pixel grid (same origin, same alignment).

In [ ]:
reference_image = ee.Image(REFERENCE_ASSET)
ref_proj        = reference_image.projection()

# CRS (e.g., 'EPSG:4326', 'EPSG:32614', etc.)
target_crs = ref_proj.crs()

# The affine transform encodes origin + pixel size + rotation.
# Using crsTransform at export time guarantees pixel-perfect alignment
# with your reference grid (vs. just matching the CRS and scale).
target_transform = ref_proj.getInfo()['transform']   # [xScale, 0, xOrigin, 0, yScale, yOrigin]

print('Target CRS       :', target_crs.getInfo())
print('Target transform :', target_transform)
print('Nominal scale (m):', ref_proj.nominalScale().getInfo())

---
## Section 5 — Masking Functions

In [ ]:
def mask_low_quality(image, count_band=COUNT_BAND):
    """
    Retain only pixels where at least MIN_MODEL_COUNT of the 6 ensemble
    members produced a valid ET estimate.

    Note: if the daily collection does not have a 'count' band, comment
    this mask out and rely on the ET range filter instead.
    """
    band_names = image.bandNames()
    has_count  = band_names.contains(count_band)

    def apply_count_mask(img):
        return img.updateMask(img.select(count_band).gte(MIN_MODEL_COUNT))

    # Only apply if the band exists (avoids errors if the collection differs)
    return ee.Image(ee.Algorithms.If(has_count, apply_count_mask(image), image))

def apply_masks(image):
    """Convenience wrapper — apply all three masks in sequence."""
    image = mask_low_quality(image)
    return image

print('Masking function defined.')

---
## Section 6 — Bimonthly Period Generator

In [ ]:
def get_bimonthly_periods(years):
    """
    Return a list of dicts, one per bimonthly period.
    Each month is split into:
      - First half : days  1–15
      - Second half: days 16–end-of-month

    Each dict has:
      'start'      : inclusive start date string (YYYY-MM-DD)
      'end'        : inclusive end date string
      'end_excl'   : exclusive end date (for GEE filterDate, which is [start, end) )
      'label'      : compact string used in filenames, e.g. '20150101_20150115'
    """
    periods = []
    for year in years:
        for month in range(1, 13):
            last_day = calendar.monthrange(year, month)[1]

            # --- First half: 1st to 15th ---
            s1 = datetime.date(year, month, 1)
            e1 = datetime.date(year, month, 15)
            periods.append({
                'start'    : s1.strftime('%Y-%m-%d'),
                'end'      : e1.strftime('%Y-%m-%d'),
                'end_excl' : (e1 + datetime.timedelta(days=1)).strftime('%Y-%m-%d'),
                'label'    : f'{s1.strftime("%Y%m%d")}_{e1.strftime("%Y%m%d")}'
            })

            # --- Second half: 16th to end of month ---
            s2 = datetime.date(year, month, 16)
            e2 = datetime.date(year, month, last_day)
            periods.append({
                'start'    : s2.strftime('%Y-%m-%d'),
                'end'      : e2.strftime('%Y-%m-%d'),
                'end_excl' : (e2 + datetime.timedelta(days=1)).strftime('%Y-%m-%d'),
                'label'    : f'{s2.strftime("%Y%m%d")}_{e2.strftime("%Y%m%d")}'
            })

    return periods


periods = get_bimonthly_periods(YEARS)
print(f'Total bimonthly periods: {len(periods)}')
print('\nFirst 6 periods:')
for p in periods[:6]:
    print(f"  {p['start']}  →  {p['end']}   label: {p['label']}")

---
## Section 7 — Processing Function

For each bimonthly period:
1. Filter the daily OpenET collection to that date window
2. Apply masks per-image before aggregating (avoids bad values pulling the median)
3. Compute pixel-wise **median** ET across all valid daily images
4. Also compute the **mean model count** as a spatial quality indicator
5. Clip to Iowa

Reprojection happens at export time (more efficient than `.reproject()` on the image).

In [ ]:
def process_period(period, collection, apply_masks=True):
    """
    Compute bimonthly median ET for one period.

    Parameters
    ----------
    period       : dict from get_bimonthly_periods()
    collection   : ee.ImageCollection — the full OpenET daily collection
    apply_masks  : bool — whether to apply water / quality / range masks

    Returns
    -------
    ee.Image with two bands:
      'et'               — median daily ET (mm/day) over the period
      'mean_model_count' — average number of models contributing (quality proxy)
    """
    # GEE filterDate uses [start, end) — 'end_excl' ensures the last day is included
    filtered = (
        collection
        .filterDate(period['start'], period['end_excl'])
        .filterBounds(iowa)
    )
    
    # Apply masks to every image before aggregating
    if apply_masks:
        filtered = filtered.map(apply_masks)

    # Pixel-wise median ET across all valid daily images in the window
    median_et = filtered.select(ET_BAND).median().rename(ET_BAND)

    # Mean model count — higher value = more models agreed = higher quality
    # (If COUNT_BAND doesn't exist in your collection, comment out these two lines)
    mean_count = filtered.select(COUNT_BAND).mean().rename('mean_model_count')

    # Combine into a single multi-band image and clip to Iowa
    result = median_et.addBands(mean_count).clip(iowa)

    # Store dates as image properties (handy metadata)
    result = result.set({
        'period_start': period['start'],
        'period_end'  : period['end'],
        'label'       : period['label']
    })

    return result


print('process_period() defined.')

---
## Section 8 — Test 

In [ ]:
openet_collection = ee.ImageCollection(OPENET_DAILY)

test_period = periods[0]   # first bimonthly period: Jan 1–15, 2015
print(f"Testing: {test_period['start']} → {test_period['end']}")

test_image = process_period(test_period, openet_collection)

print('Output bands:', test_image.bandNames().getInfo())

# Compute summary stats over Iowa (uses a coarse scale for speed)
# This call hits GEE servers — may take ~30–60 seconds
stats = test_image.select(ET_BAND).reduceRegion(
    reducer  = ee.Reducer.mean()
                         .combine(ee.Reducer.minMax(), sharedInputs=True)
                         .combine(ee.Reducer.count(),  sharedInputs=True),
    geometry = iowa,
    scale    = 500,        # coarse resolution for a quick check
    maxPixels= 1e9
)
print('ET stats for test period:')
for k, v in stats.getInfo().items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

---
## Section 9 — Export Loop

Submits one export task per bimonthly period to GEE's batch processing system.



In [ ]:
tasks = []   # keep handles so we can monitor status later

for i, period in enumerate(periods):

    image    = process_period(period, openet_collection)
    filename = f'OpenET_Iowa_{period["label"]}'

    # ── Choose which bands to export ────────────────────────────────────────
    export_image = image.select(ET_BAND)
    # To also export the quality layer:
    # export_image = image.select([ET_BAND, 'mean_model_count'])

    task = ee.batch.Export.image.toDrive(
        image          = export_image,
        description    = filename,          # task name shown in GEE task manager
        folder         = EXPORT_FOLDER,     # Google Drive folder
        fileNamePrefix = filename,          # output filename (without .tif)
        region         = iowa,              # clip/export extent

        # ── Projection: match reference raster exactly ───────────────────────
        crs            = target_crs,        # target coordinate system
        crsTransform   = target_transform,  # origin + pixel size — snaps to reference grid
        # Alternatively, use scale= if you only need the right resolution, not exact alignment:
        # scale        = ref_proj.nominalScale(),

        maxPixels      = 1e13,
        fileFormat     = 'GeoTIFF',
        formatOptions  = {'cloudOptimized': True}  # cloud-optimized GeoTIFF (optional)
    )
    task.start()
    tasks.append({'label': period['label'], 'task': task})

    print(f'[{i+1:3d}/{len(periods)}] Submitted: {filename}')

print(f'\n✓ All {len(tasks)} tasks submitted.')
print('Monitor at: https://code.earthengine.google.com/  →  Tasks tab')